In [46]:
#import librarys
import pandas as pd

import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_validate


In [2]:
df=pd.read_csv("divar.csv")


C:\Users\Amin\AppData\Local\Temp\ipykernel_8552\3341867464.py:1: DtypeWarning: Columns (11,27,29,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("divar.csv")


In [3]:
df.describe()

,Unnamed: 0,rent_value,price_value,credit_value,transformable_credit,transformed_credit,transformable_rent,transformed_rent,land_size,building_size,regular_person_capacity,cost_per_extra_person,rent_price_on_regular_days,rent_price_on_special_days,rent_price_at_weekends,location_latitude,location_longitude,location_radius
count,1000000.000000,3.513220e+05,5.683460e+05,3.520950e+05,3.520850e+05,7.240900e+04,3.512480e+05,7.240900e+04,1.863960e+05,9.803940e+05,29870.000000,1.024100e+04,1.806800e+04,1.046300e+04,1.355100e+04,655608.000000,655608.000000,339699.000000
mean,499999.500000,4.102299e+10,1.736537e+10,4.872084e+10,4.872222e+10,8.557025e+09,4.103164e+10,1.619934e+07,4.165480e+03,4.440648e+03,6.557650,1.209785e+10,1.389016e+11,2.355548e+10,3.156551e+10,34.982108,51.629743,465.149147
std,288675.278932,3.807534e+12,5.878739e+11,4.341346e+12,4.341407e+12,2.064576e+12,3.807935e+12,5.217890e+07,1.218927e+05,1.367118e+05,7.698655,1.103482e+12,7.042335e+12,1.542049e+12,2.434942e+12,2.379169,3.160920,125.896250
min,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,23.626478,40.162369,0.000000
25%,249999.750000,1.111110e+05,1.400000e+09,1.000000e+08,1.000000e+08,2.000000e+08,1.111110e+05,1.000000e+06,1.100000e+02,7.500000e+01,3.000000,5.000000e+04,4.000000e+05,6.000000e+05,5.500000e+05,34.553551,50.677175,500.000000
50%,499999.500000,5.000000e+06,2.840000e+09,2.500000e+08,2.500000e+08,4.000000e+08,5.000000e+06,6.000000e+06,1.950000e+02,1.030000e+02,4.000000,1.000000e+05,8.000000e+05,1.200000e+06,1.100000e+06,35.723312,51.345791,500.000000
75%,749999.250000,1.200000e+07,5.900000e+09,5.000000e+08,5.000000e+08,8.500000e+08,1.200000e+07,1.500000e+07,2.800000e+02,1.650000e+02,7.000000,2.000000e+05,1.600000e+06,2.500000e+06,2.500000e+06,36.307013,51.805291,500.000000
max,999999.000000,1.000000e+15,1.000000e+14,1.000000e+15,1.000000e+15,5.555556e+14,1.000000e+15,3.000000e+09,1.000000e+07,1.000000e+07,50.000000,1.111111e+14,5.006007e+14,1.111111e+14,2.002503e+14,40.358055,74.511620,500.000000


In [6]:
columns = [
    "has_elevator",
    "rooms_count",
    "has_parking",
    "has_warehouse",
    "cat2_slug",
    "cat3_slug",
    "city_slug",
    "building_size",
    "neighborhood_slug",
    "transformable_price",
    "transformed_rent",
    "transformable_rent",
    "price_mode",
    "rent_mode",
    "rent_type"
]

with open("feature_check.txt", "w", encoding="utf-8") as f:
    for column in columns:
        f.write(f"Column: {column}\n")
        f.write(f"dtype: {df[column].dtype}\n")
        f.write(f"Missing values: {df[column].isnull().sum()}\n")
        f.write(f"Unique values: {df[column].nunique()}\n")
        f.write(
            f"Sample unique values: "
            f"{df[column].dropna().unique()[:20]}\n"
        )
        f.write("-" * 50 + "\n")

In [7]:
rent_data = df[
    df["cat2_slug"].isin([
        "residential-rent",
        "commercial-rent"
    ])
]

columns = [
    "rent_type",
    "rent_mode",
    "transformable_rent",
    "transformed_rent",
    "transformed_credit"
]

with open("rent_check.txt", "w", encoding="utf-8") as f:
    f.write(f"Number of rent listings: {len(rent_data)}\n")
    f.write("=" * 60 + "\n\n")

    for column in columns:
        f.write(f"Column: {column}\n")
        f.write(f"dtype: {rent_data[column].dtype}\n")
        f.write(f"Missing values: {rent_data[column].isnull().sum()}\n")
        f.write(f"Unique values: {rent_data[column].nunique()}\n")
        f.write(
            f"Sample unique values: "
            f"{rent_data[column].dropna().unique()[:20]}\n"
        )
        f.write("-" * 50 + "\n")

In [8]:
price_columns = [
    "rent_type",
    "rent_mode",
    "rent_value",
    "transformable_rent",
    "transformed_rent",
    "transformed_credit"
]

rent_price_sample = rent_data[price_columns].dropna(
    subset=["rent_value"]
).head(30)

with open("rent_price_sample.txt", "w", encoding="utf-8") as f:
    f.write(rent_price_sample.to_string(index=False))

In [9]:
columns = [
    "rent_type",
    "rent_mode",
    "rent_value",
    "transformable_rent",
    "transformed_rent",
    "transformed_credit"
]

with open("rent_columns_analysis.txt", "w", encoding="utf-8") as f:

    f.write("=== Missing values ===\n\n")

    for column in columns:
        missing = rent_data[column].isna().sum()
        not_missing = rent_data[column].notna().sum()
        total = len(rent_data)
        missing_percent = (missing / total) * 100
        not_missing_percent = (not_missing / total) * 100

        f.write(f"Column: {column}\n")
        f.write(f"Total: {total}\n")
        f.write(f"Not missing: {not_missing} ({not_missing_percent:.2f}%)\n")
        f.write(f"Missing: {missing} ({missing_percent:.2f}%)\n")
        f.write("-" * 50 + "\n")

    f.write("\n\n=== rent_mode vs transformed columns ===\n\n")

    mode_analysis = rent_data.groupby("rent_mode")[
        ["rent_value", "transformable_rent",
         "transformed_rent", "transformed_credit"]
    ].agg(["count", "mean", "median"])

    f.write(mode_analysis.to_string())

    f.write("\n\n=== rent_type vs transformed columns ===\n\n")

    type_analysis = rent_data.groupby("rent_type")[
        ["rent_value", "transformable_rent",
         "transformed_rent", "transformed_credit"]
    ].agg(["count", "mean", "median"])

    f.write(type_analysis.to_string())

In [10]:
credit_rent = rent_data[
    ["transformed_credit", "transformed_rent"]
].dropna()

credit_rent = credit_rent[
    credit_rent["transformed_credit"] > 0
].copy()

credit_rent["rent_per_million_credit"] = (
    credit_rent["transformed_rent"]
    / credit_rent["transformed_credit"]
    * 1_000_000
)

with open("credit_rent_analysis.txt", "w", encoding="utf-8") as f:

    f.write("=== Credit / Rent Relationship ===\n\n")

    f.write(f"Number of rows: {len(credit_rent)}\n\n")

    f.write("Rent generated from 1 million credit:\n")
    f.write(
        f"Mean: "
        f"{credit_rent['rent_per_million_credit'].mean():,.2f}\n"
    )
    f.write(
        f"Median: "
        f"{credit_rent['rent_per_million_credit'].median():,.2f}\n"
    )
    f.write(
        f"Minimum: "
        f"{credit_rent['rent_per_million_credit'].min():,.2f}\n"
    )
    f.write(
        f"Maximum: "
        f"{credit_rent['rent_per_million_credit'].max():,.2f}\n"
    )

    f.write("\n\n=== Sample rows ===\n\n")

    f.write(
        credit_rent[
            [
                "transformed_credit",
                "transformed_rent",
                "rent_per_million_credit"
            ]
        ].head(50).to_string(index=False)
    )

In [11]:
analysis = rent_data[
    [
        "rent_value",
        "transformable_rent",
        "transformed_rent",
        "transformed_credit"
    ]
].copy()

analysis["rent_difference"] = (
    analysis["transformable_rent"] - analysis["rent_value"]
)

analysis["rent_ratio"] = (
    analysis["transformed_rent"] / analysis["rent_value"]
)

analysis["credit_rent_ratio"] = (
    analysis["transformed_credit"] / analysis["rent_value"]
)

with open("rent_relationship_analysis.txt", "w", encoding="utf-8") as f:

    f.write("=== Rent Relationship Analysis ===\n\n")

    f.write(f"Total rows: {len(analysis)}\n\n")

    # بررسی اختلاف rent_value و transformable_rent
    f.write("=== rent_value vs transformable_rent ===\n\n")

    equal_count = (
        analysis["rent_value"] ==
        analysis["transformable_rent"]
    ).sum()

    f.write(
        f"Equal values: {equal_count}\n"
    )

    f.write(
        f"Different values: "
        f"{len(analysis) - equal_count}\n\n"
    )

    f.write(
        "Difference statistics:\n"
    )

    f.write(
        analysis["rent_difference"]
        .describe()
        .to_string()
    )

    # بررسی transformed_rent
    f.write("\n\n=== transformed_rent ===\n\n")

    transformed_rent = analysis[
        analysis["transformed_rent"].notna()
    ]

    f.write(
        transformed_rent["transformed_rent"]
        .describe()
        .to_string()
    )

    # بررسی نسبت‌ها
    f.write("\n\n=== Ratios ===\n\n")

    f.write("transformed_rent / rent_value:\n")
    f.write(
        analysis["rent_ratio"]
        .replace([float("inf"), -float("inf")], float("nan"))
        .describe()
        .to_string()
    )

    f.write("\n\ntransformed_credit / rent_value:\n")
    f.write(
        analysis["credit_rent_ratio"]
        .replace([float("inf"), -float("inf")], float("nan"))
        .describe()
        .to_string()
    )

    # نمونه‌هایی که هر چهار مقدار را دارند
    f.write("\n\n=== Sample rows with all values ===\n\n")

    sample = analysis.dropna(
        subset=[
            "rent_value",
            "transformable_rent",
            "transformed_rent",
            "transformed_credit"
        ]
    ).head(50)

    f.write(
        sample.to_string(index=False)
    )

In [13]:
rent_values = rent_data[
    rent_data["rent_value"] > 0
]["rent_value"]

top_20_rents = rent_values.nlargest(20)

with open("top_20_rent_values.txt", "w", encoding="utf-8") as f:
    f.write("=== Top 20 Rent Values ===\n\n")
    f.write(top_20_rents.to_string(index=False))

    f.write("\n\n=== Statistics ===\n\n")
    f.write(f"Count of positive rent values: {len(rent_values)}\n")
    f.write(f"Maximum rent: {rent_values.max():,.0f}\n")
    f.write(f"Minimum positive rent: {rent_values.min():,.0f}\n")
    f.write(f"Median rent: {rent_values.median():,.0f}\n")

In [14]:
rent_values = rent_data.loc[
    rent_data["rent_value"] > 0,
    "rent_value"
]

quantiles = rent_values.quantile([
    0.90,
    0.95,
    0.99,
    0.995,
    0.999,
    0.9999
])

with open("rent_outlier_analysis.txt", "w", encoding="utf-8") as f:
    f.write("=== Rent Outlier Analysis ===\n\n")

    f.write("=== Quantiles ===\n\n")
    for q, value in quantiles.items():
        f.write(f"{q * 100:.2f}%: {value:,.0f}\n")

    f.write("\n\n=== Largest values below selected thresholds ===\n\n")

    thresholds = [
        10_000_000,
        20_000_000,
        50_000_000,
        100_000_000,
        200_000_000,
        500_000_000,
        1_000_000_000
    ]

    for threshold in thresholds:
        values = rent_values[
            rent_values <= threshold
        ].nlargest(10)

        f.write(f"\nThreshold: {threshold:,.0f}\n")
        f.write(values.to_string(index=False))
        f.write("\n")

    f.write("\n\n=== Overall statistics ===\n\n")
    f.write(rent_values.describe().to_string())

In [15]:
rent_values = rent_data.loc[
    rent_data["rent_value"] > 0,
    "rent_value"
]

bins = [
    0,
    5_000_000,
    10_000_000,
    20_000_000,
    50_000_000,
    100_000_000,
    200_000_000,
    500_000_000,
    1_000_000_000,
    5_000_000_000,
    10_000_000_000,
    50_000_000_000,
    100_000_000_000,
    1_000_000_000_000,
    float("inf")
]

labels = [
    "0 - 5M",
    "5M - 10M",
    "10M - 20M",
    "20M - 50M",
    "50M - 100M",
    "100M - 200M",
    "200M - 500M",
    "500M - 1B",
    "1B - 5B",
    "5B - 10B",
    "10B - 50B",
    "50B - 100B",
    "100B - 1T",
    "Above 1T"
]

ranges = pd.cut(
    rent_values,
    bins=bins,
    labels=labels,
    include_lowest=True
)

range_counts = ranges.value_counts().sort_index()

with open("rent_value_ranges.txt", "w", encoding="utf-8") as f:
    f.write("=== Rent Value Distribution ===\n\n")

    for label, count in range_counts.items():
        percentage = (count / len(rent_values)) * 100

        f.write(
            f"{label}: {count:,} "
            f"({percentage:.2f}%)\n"
        )

In [18]:
rent_model_data = rent_data[
    (rent_data["rent_value"] > 0) &
    (rent_data["rent_value"] <= 200_000_000)
].copy()
with open("rent_model_data_check.txt", "w", encoding="utf-8") as f:
    f.write("=== Rent Model Data Check ===\n\n")

    f.write(f"Original rent data: {len(rent_data):,}\n")
    f.write(f"Clean rent data: {len(rent_model_data):,}\n")
    f.write(
        f"Removed: {len(rent_data) - len(rent_model_data):,}\n\n"
    )

    f.write("=== Target statistics ===\n\n")
    f.write(
        rent_model_data["rent_value"]
        .describe()
        .to_string()
    )

In [19]:
low_rents = rent_model_data["rent_value"].nsmallest(50)

with open("lowest_rent_values.txt", "w", encoding="utf-8") as f:
    f.write("=== Lowest 50 Rent Values ===\n\n")
    f.write(low_rents.to_string(index=False))

In [20]:
features = [
    "has_elevator",
    "rooms_count",
    "has_parking",
    "has_warehouse",
    "cat2_slug",
    "cat3_slug",
    "city_slug",
    "building_size",
    "neighborhood_slug"
]

with open("rent_features_missing.txt", "w", encoding="utf-8") as f:
    f.write("=== Missing Values in Model Features ===\n\n")

    total = len(rent_model_data)

    for column in features:
        missing = rent_model_data[column].isna().sum()
        missing_percent = (missing / total) * 100

        f.write(
            f"{column}: "
            f"{missing:,} missing "
            f"({missing_percent:.2f}%)\n"
        )

In [21]:
categorical_features = [
    "has_elevator",
    "has_parking",
    "has_warehouse",
    "neighborhood_slug"
]

with open("rent_categorical_analysis.txt", "w", encoding="utf-8") as f:

    for column in categorical_features:

        f.write(f"=== {column} ===\n\n")

        f.write("Value counts including missing:\n\n")

        counts = rent_model_data[column].value_counts(
            dropna=False
        )

        for value, count in counts.items():
            percentage = (count / len(rent_model_data)) * 100

            f.write(
                f"{value}: "
                f"{count:,} "
                f"({percentage:.2f}%)\n"
            )

        f.write("\n" + "=" * 60 + "\n\n")

In [22]:
low_rent_analysis = {
    "1 تا 100": ((rent_model_data["rent_value"] >= 1) &
                 (rent_model_data["rent_value"] <= 100)).sum(),

    "101 تا 1000": ((rent_model_data["rent_value"] > 100) &
                    (rent_model_data["rent_value"] <= 1000)).sum(),

    "1001 تا 10000": ((rent_model_data["rent_value"] > 1000) &
                      (rent_model_data["rent_value"] <= 10000)).sum(),

    "10001 تا 50000": ((rent_model_data["rent_value"] > 10000) &
                       (rent_model_data["rent_value"] <= 50000)).sum(),

    "50001 تا 100000": ((rent_model_data["rent_value"] > 50000) &
                        (rent_model_data["rent_value"] <= 100000)).sum(),

    "100001 تا 500000": ((rent_model_data["rent_value"] > 100000) &
                         (rent_model_data["rent_value"] <= 500000)) .sum(),
}

with open("low_rent_analysis.txt", "w", encoding="utf-8") as f:
    f.write("=== Low Rent Values Analysis ===\n\n")

    for range_name, count in low_rent_analysis.items():
        f.write(f"{range_name}: {count:,}\n")

In [23]:
# Clean rental data based on rent_value

rent_model_data = rent_data[
    (rent_data["rent_value"] >= 1_000_000) &
    (rent_data["rent_value"] <= 200_000_000)
].copy()


# Calculate cleaning summary
original_count = len(rent_data)
clean_count = len(rent_model_data)
removed_count = original_count - clean_count


# Save summary to a text file
with open("rent_cleaning_summary.txt", "w", encoding="utf-8") as f:
    f.write("=== Rent Data Cleaning Summary ===\n\n")
    f.write(f"Original rent data: {original_count:,}\n")
    f.write(f"Clean rent data: {clean_count:,}\n")
    f.write(f"Removed: {removed_count:,}\n\n")

    f.write("=== Cleaning Rules ===\n")
    f.write("Minimum rent_value: 1,000,000\n")
    f.write("Maximum rent_value: 200,000,000\n")


print(f"Original rent data: {original_count:,}")
print(f"Clean rent data: {clean_count:,}")
print(f"Removed: {removed_count:,}")

Original rent data: 353,125
Clean rent data: 254,540
Removed: 98,585


In [24]:
# Check selected features after target cleaning

feature_check = {}

for column in features:
    feature_check[column] = {
        "dtype": str(rent_model_data[column].dtype),
        "missing": int(rent_model_data[column].isna().sum()),
        "missing_percent": round(
            rent_model_data[column].isna().mean() * 100, 2
        ),
        "unique": int(rent_model_data[column].nunique())
    }


with open("feature_check_after_cleaning.txt", "w", encoding="utf-8") as f:
    f.write("=== Feature Check After Rent Cleaning ===\n\n")

    for column, info in feature_check.items():
        f.write(f"Feature: {column}\n")
        f.write(f"Data type: {info['dtype']}\n")
        f.write(f"Missing: {info['missing']:,}\n")
        f.write(f"Missing percent: {info['missing_percent']}%\n")
        f.write(f"Unique values: {info['unique']:,}\n")
        f.write("-" * 50 + "\n")


print("Feature check completed.")
print("Saved to: feature_check_after_cleaning.txt")

Feature check completed.
Saved to: feature_check_after_cleaning.txt


In [25]:
categorical_features = [
    "has_elevator",
    "rooms_count",
    "has_parking",
    "has_warehouse",
    "cat2_slug",
    "cat3_slug",
    "city_slug"
]

with open("categorical_values.txt", "w", encoding="utf-8") as f:
    f.write("=== Categorical Feature Values ===\n\n")

    for column in categorical_features:
        f.write(f"### {column}\n")
        f.write(
            rent_model_data[column]
            .value_counts(dropna=False)
            .to_string()
        )
        f.write("\n\n")
        f.write("-" * 60)
        f.write("\n\n")

print("Categorical values saved to categorical_values.txt")

Categorical values saved to categorical_values.txt


In [26]:
# Copy data before preprocessing
model_data = rent_model_data.copy()


# Convert missing categorical values to "Unknown"
categorical_with_missing = [
    "has_elevator",
    "rooms_count",
    "has_parking",
    "has_warehouse"
]

for column in categorical_with_missing:
    model_data[column] = model_data[column].fillna("Unknown")


# Convert city_slug missing value to "Unknown"
model_data["city_slug"] = model_data["city_slug"].fillna("Unknown")


# Define features and target
features = [
    "has_elevator",
    "rooms_count",
    "has_parking",
    "has_warehouse",
    "cat2_slug",
    "cat3_slug",
    "city_slug",
    "building_size"
]

target = "rent_value"


# Save a summary
with open("missing_value_handling.txt", "w", encoding="utf-8") as f:
    f.write("=== Missing Value Handling ===\n\n")

    for column in features:
        missing_count = model_data[column].isna().sum()

        f.write(f"{column}: {missing_count:,} missing values remaining\n")

    f.write("\nCategorical missing values were replaced with 'Unknown'.\n")


print("Missing-value preprocessing completed.")
print("Remaining missing values:")

for column in features:
    print(f"{column}: {model_data[column].isna().sum():,}")

Missing-value preprocessing completed.
Remaining missing values:
has_elevator: 0
rooms_count: 0
has_parking: 0
has_warehouse: 0
cat2_slug: 0
cat3_slug: 0
city_slug: 0
building_size: 0


In [28]:
print("=== Building Size Statistics ===")

print(f"Count: {model_data['building_size'].count():,}")
print(f"Minimum: {model_data['building_size'].min():,.2f}")
print(f"Maximum: {model_data['building_size'].max():,.2f}")
print(f"Median: {model_data['building_size'].median():,.2f}")

print("\n=== Percentiles ===")

percentiles = model_data["building_size"].quantile(
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999]
)

print(percentiles)

=== Building Size Statistics ===
Count: 254,540
Minimum: 1.00
Maximum: 10,000,000.00
Median: 90.00

=== Percentiles ===
0.010        11.000000
0.050        20.000000
0.250        65.000000
0.500        90.000000
0.750       130.000000
0.900       180.000000
0.950       250.000000
0.990      2000.000000
0.995     35000.000000
0.999    763830.000001
Name: building_size, dtype: float64


In [29]:
size_ranges = [
    (0, 50),
    (50, 100),
    (100, 200),
    (200, 500),
    (500, 1000),
    (1000, 2000),
    (2000, 5000),
    (5000, 10000),
    (10000, 50000),
    (50000, float("inf"))
]

print("=== Building Size Distribution ===\n")

with open("building_size_distribution.txt", "w", encoding="utf-8") as f:

    f.write("=== Building Size Distribution ===\n\n")

    for lower, upper in size_ranges:
        count = (
            (model_data["building_size"] > lower) &
            (model_data["building_size"] <= upper)
        ).sum()

        if upper == float("inf"):
            range_text = f"{lower:,} < size"
        else:
            range_text = f"{lower:,} < size <= {upper:,}"

        result = f"{range_text}: {count:,}"

        print(result)
        f.write(result + "\n")

=== Building Size Distribution ===

0 < size <= 50: 42,246
50 < size <= 100: 111,748
100 < size <= 200: 81,902
200 < size <= 500: 13,076
500 < size <= 1,000: 2,032
1,000 < size <= 2,000: 1,050
2,000 < size <= 5,000: 630
5,000 < size <= 10,000: 239
10,000 < size <= 50,000: 442
50,000 < size: 1,175


In [30]:
large_size_data = model_data[
    model_data["building_size"] > 2_000
]

print("=== Building Size > 2,000 Analysis ===\n")

with open("large_building_size_analysis.txt", "w", encoding="utf-8") as f:

    f.write("=== Building Size > 2,000 Analysis ===\n\n")

    f.write(
        f"Total records above 2,000 m²: "
        f"{len(large_size_data):,}\n\n"
    )

    # Analysis by property type
    f.write("=== By cat3_slug ===\n\n")

    cat3_counts = large_size_data["cat3_slug"].value_counts()

    for category, count in cat3_counts.items():
        f.write(f"{category}: {count:,}\n")

    # Analysis by cat2_slug
    f.write("\n=== By cat2_slug ===\n\n")

    cat2_counts = large_size_data["cat2_slug"].value_counts()

    for category, count in cat2_counts.items():
        f.write(f"{category}: {count:,}\n")

    # Some statistics
    f.write("\n=== Statistics ===\n\n")

    f.write(
        f"Minimum: {large_size_data['building_size'].min():,.2f}\n"
    )

    f.write(
        f"Median: {large_size_data['building_size'].median():,.2f}\n"
    )

    f.write(
        f"Maximum: {large_size_data['building_size'].max():,.2f}\n"
    )

print("Analysis saved to large_building_size_analysis.txt")

=== Building Size > 2,000 Analysis ===

Analysis saved to large_building_size_analysis.txt


In [31]:
large_size_data = model_data[
    model_data["building_size"] > 2_000
]

size_ranges_large = [
    (2_000, 5_000),
    (5_000, 10_000),
    (10_000, 50_000),
    (50_000, float("inf"))
]

print("=== Large Building Size by Property Type ===\n")

with open(
    "large_building_size_by_type.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write("=== Large Building Size by Property Type ===\n\n")

    for category in large_size_data["cat3_slug"].unique():

        category_data = large_size_data[
            large_size_data["cat3_slug"] == category
        ]

        f.write(f"=== {category} ===\n")

        print(f"=== {category} ===")

        for lower, upper in size_ranges_large:

            if upper == float("inf"):
                condition = category_data["building_size"] > lower
                range_text = f"> {lower:,}"
            else:
                condition = (
                    (category_data["building_size"] > lower) &
                    (category_data["building_size"] <= upper)
                )
                range_text = (
                    f"{lower:,} < size <= {upper:,}"
                )

            count = condition.sum()

            result = f"{range_text}: {count:,}"

            print(result)
            f.write(result + "\n")

        print()
        f.write("\n")

=== Large Building Size by Property Type ===

=== house-villa-rent ===
2,000 < size <= 5,000: 16
5,000 < size <= 10,000: 24
10,000 < size <= 50,000: 119
> 50,000: 395

=== industry-agriculture-business-rent ===
2,000 < size <= 5,000: 548
5,000 < size <= 10,000: 190
10,000 < size <= 50,000: 210
> 50,000: 341

=== office-rent ===
2,000 < size <= 5,000: 20
5,000 < size <= 10,000: 3
10,000 < size <= 50,000: 13
> 50,000: 47

=== shop-rent ===
2,000 < size <= 5,000: 35
5,000 < size <= 10,000: 10
10,000 < size <= 50,000: 47
> 50,000: 122

=== apartment-rent ===
2,000 < size <= 5,000: 11
5,000 < size <= 10,000: 12
10,000 < size <= 50,000: 53
> 50,000: 270



In [32]:
building_size_clean_data = model_data[
    (model_data["building_size"] >= 1) &
    (model_data["building_size"] <= 10_000)
].copy()

original_count = len(model_data)
clean_count = len(building_size_clean_data)
removed_count = original_count - clean_count

removed_percent = (removed_count / original_count) * 100

print("=== Building Size Cleaning Summary ===\n")
print(f"Original data: {original_count:,}")
print(f"Clean data: {clean_count:,}")
print(f"Removed: {removed_count:,}")
print(f"Removed percentage: {removed_percent:.2f}%")

with open(
    "building_size_cleaning_summary.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write("=== Building Size Cleaning Summary ===\n\n")
    f.write(f"Original data: {original_count:,}\n")
    f.write(f"Clean data: {clean_count:,}\n")
    f.write(f"Removed: {removed_count:,}\n")
    f.write(f"Removed percentage: {removed_percent:.2f}%\n\n")

    f.write("=== Cleaning Rules ===\n")
    f.write("Minimum building_size: 1 m²\n")
    f.write("Maximum building_size: 10,000 m²\n")

=== Building Size Cleaning Summary ===

Original data: 254,540
Clean data: 252,923
Removed: 1,617
Removed percentage: 0.64%


In [33]:
model_data = building_size_clean_data.copy()

print(f"Final model data: {len(model_data):,} rows")

Final model data: 252,923 rows


In [34]:
# Features
X = model_data[features]

# Target
y = model_data[target]

# Split data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("=== Data Split ===")
print(f"X_train: {X_train.shape}")
print(f"X_valid: {X_valid.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_valid: {y_valid.shape}")

=== Data Split ===
X_train: (202338, 8)
X_valid: (50585, 8)
y_train: (202338,)
y_valid: (50585,)


In [35]:

# Categorical features
categorical_features = [
    "has_elevator",
    "rooms_count",
    "has_parking",
    "has_warehouse",
    "cat2_slug",
    "cat3_slug",
    "city_slug"
]

# Numerical features
numerical_features = [
    "building_size"
]

# Create preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [37]:
# Convert all categorical features to string
for column in categorical_features:
    X_train[column] = X_train[column].astype(str)
    X_valid[column] = X_valid[column].astype(str)

print("Categorical features converted to string.")

Categorical features converted to string.


In [38]:
# Fit the preprocessor only on training data
preprocessor.fit(X_train)

# Transform training and validation data
X_train_encoded = preprocessor.transform(X_train)
X_valid_encoded = preprocessor.transform(X_valid)

print("=== Encoding Completed ===")
print(f"X_train_encoded shape: {X_train_encoded.shape}")
print(f"X_valid_encoded shape: {X_valid_encoded.shape}")

=== Encoding Completed ===
X_train_encoded shape: (202338, 430)
X_valid_encoded shape: (50585, 430)


In [39]:
print("Type of encoded training data:")
print(type(X_train_encoded))

print("\nType of encoded validation data:")
print(type(X_valid_encoded))

Type of encoded training data:
<class 'scipy.sparse._csr.csr_matrix'>

Type of encoded validation data:
<class 'scipy.sparse._csr.csr_matrix'>


In [41]:

# Create the model
linear_model = LinearRegression()

# Train the model
linear_model.fit(X_train_encoded, y_train)

print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


In [42]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Predict on validation data
y_valid_pred = linear_model.predict(X_valid_encoded)

# Calculate evaluation metrics
linear_r2 = r2_score(y_valid, y_valid_pred)
linear_mae = mean_absolute_error(y_valid, y_valid_pred)
linear_mse = mean_squared_error(y_valid, y_valid_pred)

print("=== Linear Regression - Validation Results ===")
print(f"R²:  {linear_r2:.4f}")
print(f"MAE: {linear_mae:,.2f}")
print(f"MSE: {linear_mse:,.2f}")

=== Linear Regression - Validation Results ===
R²:  0.3011
MAE: 10,837,033.63
MSE: 407,239,411,877,504.31


In [45]:
# Create Decision Tree Regression model
tree_model = DecisionTreeRegressor(
    random_state=42
)

# Train the model
tree_model.fit(X_train_encoded, y_train)

print("Decision Tree Regression model trained successfully.")

# Predict on validation data
y_valid_pred_tree = tree_model.predict(X_valid_encoded)

# Calculate evaluation metrics
tree_r2 = r2_score(y_valid, y_valid_pred_tree)
tree_mae = mean_absolute_error(y_valid, y_valid_pred_tree)
tree_mse = mean_squared_error(y_valid, y_valid_pred_tree)

print("=== Decision Tree Regression - Validation Results ===")
print(f"R²:  {tree_r2:.4f}")
print(f"MAE: {tree_mae:,.2f}")
print(f"MSE: {tree_mse:,.2f}")

with open(
    "decision_tree_validation_results.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write("=== Decision Tree Regression - Validation Results ===\n\n")
    f.write(f"R²:  {tree_r2:.4f}\n")
    f.write(f"MAE: {tree_mae:,.2f}\n")
    f.write(f"MSE: {tree_mse:,.2f}\n")

print("Results saved to decision_tree_validation_results.txt")

Decision Tree Regression model trained successfully.
=== Decision Tree Regression - Validation Results ===
R²:  0.3638
MAE: 8,945,592.79
MSE: 370,735,028,265,345.81
Results saved to decision_tree_validation_results.txt


In [47]:
# Create Decision Tree pipeline
tree_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])

# Perform 5-Fold Cross-Validation on training data
cv_results = cross_validate(
    tree_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring=[
        "r2",
        "neg_mean_absolute_error",
        "neg_mean_squared_error"
    ],
    n_jobs=-1
)

# Convert negative error scores to positive values
cv_r2 = cv_results["test_r2"]
cv_mae = -cv_results["test_neg_mean_absolute_error"]
cv_mse = -cv_results["test_neg_mean_squared_error"]

# Calculate mean and standard deviation
cv_r2_mean = cv_r2.mean()
cv_r2_std = cv_r2.std()

cv_mae_mean = cv_mae.mean()
cv_mae_std = cv_mae.std()

cv_mse_mean = cv_mse.mean()
cv_mse_std = cv_mse.std()

# Print results
print("=== Decision Tree - 5-Fold Cross-Validation ===")
print()
print("R²:")
print(f"Mean: {cv_r2_mean:.4f}")
print(f"Std:  {cv_r2_std:.4f}")

print()
print("MAE:")
print(f"Mean: {cv_mae_mean:,.2f}")
print(f"Std:  {cv_mae_std:,.2f}")

print()
print("MSE:")
print(f"Mean: {cv_mse_mean:,.2f}")
print(f"Std:  {cv_mse_std:,.2f}")

=== Decision Tree - 5-Fold Cross-Validation ===

R²:
Mean: 0.3533
Std:  0.0102

MAE:
Mean: 9,066,631.76
Std:  91,116.98

MSE:
Mean: 384,340,499,372,961.69
Std:  9,406,492,154,559.82


In [48]:
# Values of max_depth to evaluate
max_depth_values = [5, 10, 15, 20, 25, 30]

depth_results = []

for depth in max_depth_values:

    tree_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeRegressor(
                max_depth=depth,
                random_state=42
            )
        )
    ])

    cv_results = cross_validate(
        tree_pipeline,
        X_train,
        y_train,
        cv=5,
        scoring=[
            "r2",
            "neg_mean_absolute_error",
            "neg_mean_squared_error"
        ],
        n_jobs=-1
    )

    r2_mean = cv_results["test_r2"].mean()
    mae_mean = -cv_results["test_neg_mean_absolute_error"].mean()
    mse_mean = -cv_results["test_neg_mean_squared_error"].mean()

    depth_results.append({
        "max_depth": depth,
        "r2_mean": r2_mean,
        "mae_mean": mae_mean,
        "mse_mean": mse_mean
    })

# Print results
print("=== Decision Tree - max_depth Comparison ===")
print()

for result in depth_results:
    print(f"max_depth: {result['max_depth']}")
    print(f"R²:  {result['r2_mean']:.4f}")
    print(f"MAE: {result['mae_mean']:,.2f}")
    print(f"MSE: {result['mse_mean']:,.2f}")
    print("-" * 40)

c:\Users\Amin\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


=== Decision Tree - max_depth Comparison ===

max_depth: 5
R²:  0.4519
MAE: 9,238,658.07
MSE: 325,769,696,898,041.62
----------------------------------------
max_depth: 10
R²:  0.4857
MAE: 8,578,035.11
MSE: 305,711,224,356,232.19
----------------------------------------
max_depth: 15
R²:  0.4463
MAE: 8,623,263.00
MSE: 329,124,472,235,792.38
----------------------------------------
max_depth: 20
R²:  0.4092
MAE: 8,736,700.33
MSE: 351,134,827,827,398.19
----------------------------------------
max_depth: 25
R²:  0.3920
MAE: 8,798,060.33
MSE: 361,348,008,599,791.69
----------------------------------------
max_depth: 30
R²:  0.3788
MAE: 8,855,224.47
MSE: 369,186,796,685,000.25
----------------------------------------


In [49]:
# Create the tuned Decision Tree model
tuned_tree_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        DecisionTreeRegressor(
            max_depth=10,
            random_state=42
        )
    )
])

# Train the model on the complete training data
tuned_tree_model.fit(X_train, y_train)

print("Tuned Decision Tree model trained successfully.")

Tuned Decision Tree model trained successfully.


In [50]:
# Predict on validation data
y_valid_pred_tuned_tree = tuned_tree_model.predict(X_valid)

# Calculate evaluation metrics
tuned_tree_r2 = r2_score(y_valid, y_valid_pred_tuned_tree)
tuned_tree_mae = mean_absolute_error(y_valid, y_valid_pred_tuned_tree)
tuned_tree_mse = mean_squared_error(y_valid, y_valid_pred_tuned_tree)

print("=== Tuned Decision Tree - Validation Results ===")
print(f"R²:  {tuned_tree_r2:.4f}")
print(f"MAE: {tuned_tree_mae:,.2f}")
print(f"MSE: {tuned_tree_mse:,.2f}")

=== Tuned Decision Tree - Validation Results ===
R²:  0.5018
MAE: 8,452,925.50
MSE: 290,291,888,179,575.94
